# Negative DDI Sampling
## Generate Drug Pairs Without Documented Interactions

This notebook creates negative samples - drug pairs that DO NOT have documented DDIs in the adverse-only DrugBank dataset. These are used for balanced training data in machine learning models.

In [1]:
# Import Required Libraries
import pandas as pd
import numpy as np
from itertools import combinations
import random
from datetime import datetime

print("✓ Libraries imported successfully")

✓ Libraries imported successfully


In [3]:
# Load DrugBank DDI Data (Adverse-Only with Positive Removed)
adverse_ddi_path = r'C:\Users\ashto\ddi-prediction\data\sample\drugbank_approved_small_1113772_ddi_pairs_positive_removed.csv'

print(f"Loading adverse-only DDI dataset...")
ddi_df = pd.read_csv(adverse_ddi_path)

print(f"✓ Loaded {len(ddi_df):,} adverse DDI pairs")
print(f"\nColumns: {ddi_df.columns.tolist()}")
print(f"Data shape: {ddi_df.shape}")

Loading adverse-only DDI dataset...
✓ Loaded 1,113,772 adverse DDI pairs

Columns: ['drug1_id', 'drug1_name', 'drug2_id', 'drug2_name', 'description', 'pair_key', 'is_positive', 'pattern_type', 'is_positive_v2', 'is_therapeutic_efficacy']
Data shape: (1113772, 10)


In [4]:
# Extract Approved Drugs List
# Get unique drugs from drug1 and drug2 columns

drugs_from_col1 = ddi_df['drug1_id'].unique().tolist()
drugs_from_col2 = ddi_df['drug2_id'].unique().tolist()

# Combine and get unique drugs
all_drugs = list(set(drugs_from_col1 + drugs_from_col2))
all_drugs.sort()

print(f"✓ Extracted unique drugs")
print(f"  Total unique drugs: {len(all_drugs):,}")

# Also get drug names mapping
drug_mapping = {}
for idx, row in ddi_df.iterrows():
    drug_mapping[row['drug1_id']] = row['drug1_name']
    drug_mapping[row['drug2_id']] = row['drug2_name']

print(f"  Drug names mapped: {len(drug_mapping):,}")

✓ Extracted unique drugs
  Total unique drugs: 4,527
  Drug names mapped: 4,527


In [5]:
# Identify Drug Pairs Without DDIs
# Create a set of documented DDI pairs for fast lookup

print("Creating documented DDI pair set...")

# Create normalized pairs (both directions) for comparison
documented_pairs = set()

for idx, row in ddi_df.iterrows():
    drug1 = str(row['drug1_id'])
    drug2 = str(row['drug2_id'])
    
    # Store both directions since DDIs can work both ways
    pair1 = tuple(sorted([drug1, drug2]))
    documented_pairs.add(pair1)

print(f"✓ Documented DDI pairs: {len(documented_pairs):,}")

# Calculate total possible pairs
total_possible_pairs = len(list(combinations(all_drugs, 2)))
print(f"  Total possible drug pairs: {total_possible_pairs:,}")
print(f"  Potentially available negative pairs: {total_possible_pairs - len(documented_pairs):,}")

Creating documented DDI pair set...
✓ Documented DDI pairs: 1,113,772
  Total possible drug pairs: 10,244,601
  Potentially available negative pairs: 9,130,829


In [6]:
# Generate Negative Sample Pairs
# Strategy: Sample negative pairs to match the size of documented DDIs

target_negative_samples = len(documented_pairs)  # Match positive class size
print(f"Generating {target_negative_samples:,} negative samples...")
print(f"(This may take a minute...)\n")

negative_pairs = []
random.seed(42)  # For reproducibility

# Generate random pairs until we have enough that are NOT in documented_pairs
attempts = 0
max_attempts = target_negative_samples * 10  # Safety limit

while len(negative_pairs) < target_negative_samples and attempts < max_attempts:
    # Randomly select 2 drugs
    drug_pair = random.sample(all_drugs, 2)
    pair_key = tuple(sorted(drug_pair))
    
    # Check if this pair is NOT in documented DDIs
    if pair_key not in documented_pairs:
        negative_pairs.append(pair_key)
        
        if len(negative_pairs) % 100000 == 0:
            print(f"  Generated {len(negative_pairs):,} negative pairs...")
    
    attempts += 1

print(f"✓ Successfully generated {len(negative_pairs):,} negative pairs")
print(f"  Attempts needed: {attempts:,}")

if len(negative_pairs) < target_negative_samples:
    print(f"  ⚠️ Warning: Could only generate {len(negative_pairs):,} / {target_negative_samples:,} samples")
    print(f"     (Density of documented DDIs may be high)")

Generating 1,113,772 negative samples...
(This may take a minute...)

  Generated 100,000 negative pairs...
  Generated 200,000 negative pairs...
  Generated 300,000 negative pairs...
  Generated 400,000 negative pairs...
  Generated 500,000 negative pairs...
  Generated 600,000 negative pairs...
  Generated 700,000 negative pairs...
  Generated 800,000 negative pairs...
  Generated 900,000 negative pairs...
  Generated 1,000,000 negative pairs...
  Generated 1,100,000 negative pairs...
✓ Successfully generated 1,113,772 negative pairs
  Attempts needed: 1,250,447


In [7]:
# Validate and Format Negative Samples
print("Validating and formatting negative samples...\n")

# Create dataframe for negative samples
negative_data = []

for drug1_id, drug2_id in negative_pairs:
    negative_data.append({
        'drug1_id': drug1_id,
        'drug1_name': drug_mapping.get(drug1_id, 'Unknown'),
        'drug2_id': drug2_id,
        'drug2_name': drug_mapping.get(drug2_id, 'Unknown'),
        'description': 'No documented interaction',
        'pair_key': (drug1_id, drug2_id),
        'is_negative': True
    })

negative_df = pd.DataFrame(negative_data)

print(f"✓ Formatted {len(negative_df):,} negative samples")
print(f"\nDataFrame shape: {negative_df.shape}")
print(f"Columns: {negative_df.columns.tolist()}")

# Validation checks
print(f"\n--- Validation Checks ---")
print(f"✓ No null values in drug IDs: {negative_df[['drug1_id', 'drug2_id']].isnull().sum().sum() == 0}")
print(f"✓ No duplicate pairs: {len(negative_df) == len(negative_df.drop_duplicates(subset=['drug1_id', 'drug2_id']))}")

# Verify no overlap with documented DDIs
overlap_count = 0
for idx, row in negative_df.iterrows():
    pair = tuple(sorted([row['drug1_id'], row['drug2_id']]))
    if pair in documented_pairs:
        overlap_count += 1

print(f"✓ No overlap with documented DDIs: {overlap_count == 0} (overlaps: {overlap_count})")

print(f"\nSample negative pairs:")
print(negative_df.head(10))

Validating and formatting negative samples...

✓ Formatted 1,113,772 negative samples

DataFrame shape: (1113772, 7)
Columns: ['drug1_id', 'drug1_name', 'drug2_id', 'drug2_name', 'description', 'pair_key', 'is_negative']

--- Validation Checks ---
✓ No null values in drug IDs: True
✓ No duplicate pairs: False
✓ No overlap with documented DDIs: True (overlaps: 0)

Sample negative pairs:
  drug1_id                           drug1_name drug2_id  \
0  DB06608                          Tafenoquine  DB08899   
1  DB01260                             Desonide  DB06043   
2  DB00803                             Colistin  DB13108   
3  DB00862                           Vardenafil  DB05679   
4  DB06264                          Tolperisone  DB14715   
5  DB00276                            Amsacrine  DB04863   
6  DB13044                             Gossypol  DB16783   
7  DB05814                             GPI-1485  DB13508   
8  DB00066                          Follitropin  DB08934   
9  DB01537 

In [9]:
# Save Negative Samples to File
output_path = r'C:\Users\ashto\ddi-prediction\data\sample\negative_ddi_samples.csv'

# Remove pair_key column before saving (it's internal use only)
negative_df_export = negative_df.drop(columns=['pair_key', 'is_negative'])
negative_df_export.to_csv(output_path, index=False)

print(f"✓ Saved {len(negative_df_export):,} negative samples")
print(f"  File: {output_path}")

print(f"\n{'='*80}")
print(f"SUMMARY")
print(f"{'='*80}")
print(f"Negative samples created: {len(negative_df):,}")
print(f"Documented adverse DDIs: {len(ddi_df):,}")
print(f"Class balance: {len(negative_df):,} negative : {len(ddi_df):,} adverse")
print(f"Total training pairs: {len(negative_df) + len(ddi_df):,}")
print(f"\nReady for machine learning model training!")

✓ Saved 1,113,772 negative samples
  File: C:\Users\ashto\ddi-prediction\data\sample\negative_ddi_samples.csv

SUMMARY
Negative samples created: 1,113,772
Documented adverse DDIs: 1,113,772
Class balance: 1,113,772 negative : 1,113,772 adverse
Total training pairs: 2,227,544

Ready for machine learning model training!
